In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1425_Major_Dhyan_Chand_National_Stadium_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,173.91,259.59,4.95,3.30,8.20,50.63,4.05,1.37,7.36,...,NaN,10.73,75.02,0.83,144.81,0.0,0.0,38.23,987.63,NaN
1,2024-01-02,181.09,275.94,10.18,3.17,13.34,56.43,3.70,1.59,8.87,...,NaN,10.36,72.50,0.68,123.80,0.0,0.0,52.67,986.92,NaN
2,2024-01-03,186.42,264.86,22.72,26.90,34.79,56.94,1.90,2.03,8.03,...,NaN,9.72,84.26,0.85,101.49,0.0,0.0,32.82,986.81,NaN
3,2024-01-04,226.35,330.76,16.92,50.46,40.69,53.73,4.22,1.41,3.24,...,NaN,10.16,82.69,0.76,230.73,0.0,0.0,17.40,987.06,NaN
4,2024-01-05,174.19,289.28,14.02,46.62,36.20,56.67,4.57,1.65,2.99,...,NaN,10.76,85.81,0.87,163.80,0.0,0.0,12.15,987.05,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,252.71,285.43,46.59,65.63,72.36,63.87,6.00,0.83,7.13,...,NaN,16.06,88.29,0.93,147.42,0.0,0.0,7.02,994.34,NaN
362,2024-12-28,125.93,147.87,28.73,58.05,54.24,51.76,6.42,1.03,3.43,...,NaN,16.35,89.40,0.72,170.29,0.0,0.0,14.44,993.79,NaN
363,2024-12-29,114.54,133.33,3.34,33.78,20.68,52.06,6.71,0.64,18.21,...,NaN,16.18,80.94,0.98,229.86,0.0,0.0,66.38,995.46,NaN
364,2024-12-30,106.88,124.00,4.89,33.60,21.18,49.54,6.94,0.72,26.86,...,NaN,14.90,77.10,0.92,226.54,0.0,0.0,69.67,995.06,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 3
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (363, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         173.91        259.59        4.95         3.30   
1  2024-01-02         181.09        275.94       10.18         3.17   
2  2024-01-03         186.42        264.86       22.72        26.90   
3  2024-01-04         226.35        330.76       16.92        50.46   
4  2024-01-05         174.19        289.28       14.02        46.62   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0       8.20        50.63         4.05        1.37           7.36   
1      13.34        56.43         3.70        1.59           8.87   
2      34.79        56.94         1.90        2.03           8.03   
3      40.69        53.73         4.22        1.41           3.24   
4      36.20        56.67         4.57        1.65           2.99   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             1.42             2.74    10.73   75.02      0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.223110,0.827185,-1.018046,-2.085239,-1.367870,1.198027,-1.432105,0.133343,-1.148904,-0.355844,-1.447828,-1.927529,0.954837,-0.630653,-1.193359,0.0,0.0,-1.778850,0.924291
1,2024-01-02,1.333316,0.997373,-0.847346,-2.090591,-1.229493,1.634454,-1.514283,0.555747,-1.085206,-0.219941,-1.307375,-1.974394,0.766588,-1.276488,-1.810645,0.0,0.0,-1.425190,0.807141
2,2024-01-03,1.415127,0.882041,-0.438057,-1.113643,-0.652021,1.672829,-1.936912,1.400553,-1.120641,2.090411,0.191486,-2.055460,1.645083,-0.544542,-2.466125,0.0,0.0,-1.911350,0.788991
3,2024-01-04,2.028016,1.567996,-0.627361,-0.143694,-0.493183,1.431290,-1.392191,0.210144,-1.322704,3.231997,1.090803,-1.999727,1.527801,-0.932043,1.331020,0.0,0.0,-2.289011,0.830241
4,2024-01-05,1.227408,1.136229,-0.722013,-0.301785,-0.614062,1.652513,-1.310013,0.670947,-1.333250,3.159516,0.912616,-1.923729,1.760872,-0.458430,-0.635422,0.0,0.0,-2.417592,0.828591
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
358,2024-12-27,2.432618,1.096155,0.341027,0.480844,0.359428,2.194284,-0.974258,-0.903465,-1.158606,-0.020616,-0.345169,-1.252407,1.946133,-0.200096,-1.116675,0.0,0.0,-2.543235,2.031442
359,2024-12-28,0.486661,-0.335711,-0.241899,0.168781,-0.128394,1.283055,-0.875644,-0.519462,-1.314689,-0.537048,-0.382903,-1.215675,2.029052,-1.104266,-0.444742,0.0,0.0,-2.361506,1.940692
360,2024-12-29,0.311835,-0.487058,-1.070594,-0.830398,-1.031887,1.305629,-0.807554,-1.268268,-0.691204,-0.772613,-1.112418,-1.237208,1.397073,0.015182,1.305459,0.0,0.0,-1.089409,2.216242
361,2024-12-30,0.194261,-0.584174,-1.020004,-0.837809,-1.018426,1.116009,-0.753552,-1.114667,-0.326309,-0.799794,-1.101937,-1.399338,1.110217,-0.243152,1.207915,0.0,0.0,-1.008831,2.150242


In [10]:
df.to_excel('majordhyanchand2024.xlsx', index=False)